# Favorita store sales: exploratory analysis

This notebook is the data-understanding step for the demand-forecasting project. The modelling pipeline lives in `scripts/run_pipeline.py`. The questions here are simple: what is in each file, which days are missing, and whether holidays, promotions, oil prices, and paydays show up in sales.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from config import DATA_DIR, N_FAMILIES, N_STORES
from src.data import choose_subset, load_raw, profile_raw
from src.plots import run_eda

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
raw = load_raw(DATA_DIR)
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /workspace/data/raw


## Holiday file

Check the columns rather than assuming them. `transferred == True` means the holiday was moved off that date.

In [2]:
holidays = raw["holidays"]
display(holidays.head())
holidays.info()
print("transferred counts")
display(holidays["transferred"].value_counts())
print("type counts")
display(holidays["type"].value_counts())

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[us]
 1   type         350 non-null    str           
 2   locale       350 non-null    str           
 3   locale_name  350 non-null    str           
 4   description  350 non-null    str           
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[us](1), str(4)
memory usage: 28.7 KB
transferred counts


transferred
False    338
True      12
Name: count, dtype: int64

type counts


type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64

## Shapes, missing values, and the calendar

In [3]:
profile = profile_raw(raw)
for name, info in profile.items():
    if not isinstance(info, dict) or "shape" not in info:
        continue
    print(f"{name:15} shape={info['shape']} duplicate_rows={info['duplicate_rows']} missing={info['missing']}")
print("train dates", profile["train"]["date_min"], "->", profile["train"]["date_max"])
print("missing days", profile["train"]["missing_days"])
print("duplicate store-family-date keys", profile["train"]["duplicate_keys"])
print("oil null prices", profile["oil"]["null_prices"], "dates absent from the oil file", profile["oil"]["dates_absent_from_file"])
print("official test dates (no sales labels)", profile["test"]["date_min"], "->", profile["test"]["date_max"])

train           shape=[3000888, 6] duplicate_rows=0 missing={'id': 0, 'date': 0, 'store_nbr': 0, 'family': 0, 'sales': 0, 'onpromotion': 0}
test            shape=[28512, 5] duplicate_rows=0 missing={'id': 0, 'date': 0, 'store_nbr': 0, 'family': 0, 'onpromotion': 0}
stores          shape=[54, 5] duplicate_rows=0 missing={'store_nbr': 0, 'city': 0, 'state': 0, 'store_type': 0, 'cluster': 0}
oil             shape=[1218, 2] duplicate_rows=0 missing={'date': 0, 'dcoilwtico': 43}
transactions    shape=[83488, 3] duplicate_rows=0 missing={'date': 0, 'store_nbr': 0, 'transactions': 0}
holidays        shape=[350, 6] duplicate_rows=0 missing={'date': 0, 'type': 0, 'locale': 0, 'locale_name': 0, 'description': 0, 'transferred': 0}
train dates 2013-01-01 -> 2017-08-15
missing days ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']
duplicate store-family-date keys 0
oil null prices 43 dates absent from the oil file 486
official test dates (no sales labels) 2017-08-16 -> 2017-08-31


## Charts

Totals are easier to read than 3 million raw rows. Store type and cluster are compared as mean sales per store, so a type with more stores does not automatically look larger.

In [4]:
stats = run_eda(raw, ROOT / "artifacts" / "figures")
stats_view = {k: v for k, v in stats.items() if k not in ("dow_means", "month_means", "day_of_month_means")}
stats_view

{'total_sales': 1073644928.0,
 'top5_family_share': 0.7871604561805725,
 'busiest_dow': 'Sunday',
 'quietest_dow': 'Thursday',
 'weekend_vs_weekday_ratio': 1.3936416372821077,
 'peak_month': 12,
 'quiet_month': 2,
 'store_type_mean_sales': {'C': 10962317.0,
  'E': 14955609.0,
  'B': 18157580.0,
  'D': 19504628.0,
  'A': 39227092.0},
 'holiday_means': {'ordinary day': 11691.1455078125,
  'any matched holiday': 13974.505859375,
  'national holiday': 14006.7568359375,
  'regional holiday': 7580.939453125,
  'local holiday': 14094.796875,
  'event': 14025.8076171875},
 'promo_means': {'not on promotion': 158.24667358398438,
  'on promotion': 1137.6937255859375},
 'oil_weekly_corr': -0.7555697899667674,
 'oil_weekly_diff_corr': 0.03109660729678307,
 'payday_mean': 11962.33203125,
 'day_after_payday_mean': 12571.5078125,
 'other_day_mean': 11795.6103515625,
 'earthquake_date': '2016-04-16',
 'sales_14d_before_earthquake': 793002.8125,
 'sales_14d_after_earthquake': 922941.625,
 'transactions

## Findings

These sentences are filled from the summary above.

In [5]:
stores, families, family_sales, store_sales = choose_subset(raw["train"], N_STORES, N_FAMILIES)
print(
    f"1. The training panel has {profile['train']['shape'][0]:,} rows, "
    f"{profile['train']['n_stores']} stores and {profile['train']['n_families']} families, "
    f"from {profile['train']['date_min']} to {profile['train']['date_max']}. "
    f"The only missing dates are {', '.join(profile['train']['missing_days'])}. "
    "Those are Christmas days, when the stores were closed, so the dates are absent rather than stored as zero."
)
print(
    f"2. Sales are highly concentrated. The five largest families account for "
    f"{stats['top5_family_share']:.0%} of unit sales. The busiest weekday in the company total is "
    f"{stats['busiest_dow']} and the quietest is {stats['quietest_dow']}. "
    f"Weekend days average {stats['weekend_vs_weekday_ratio']:.2f} times a weekday. "
    f"The strongest month is month {stats['peak_month']} and the weakest is month {stats['quiet_month']}."
)
print(
    "3. Store type and cluster are not just labels. Mean total sales per store by type: "
    + ", ".join(f"{k}={v:,.0f}" for k, v in stats["store_type_mean_sales"].items())
    + "."
)
h = stats["holiday_means"]
print(
    f"4. A matched holiday is not automatically a high-sales day. "
    f"Mean store-day sales are {h['ordinary day']:,.0f} on ordinary days and "
    f"{h['national holiday']:,.0f} on national holidays "
    f"({h['national holiday'] / h['ordinary day'] - 1:+.1%} versus an ordinary day). "
    "Christmas itself is missing from the file, so this comparison does not include the closure."
)
p = stats["promo_means"]
print(
    f"5. Promotions line up with higher sales, but that is not a clean causal effect: "
    f"promoted rows average {p.get('on promotion', float('nan')):,.1f} units and "
    f"other rows average {p.get('not on promotion', float('nan')):,.1f}. "
    "Popular families are promoted more often."
)
print(
    f"6. Weekly oil prices and weekly total sales have a correlation of {stats['oil_weekly_corr']:.2f}. "
    f"Week-to-week changes are almost uncorrelated ({stats['oil_weekly_diff_corr']:.2f}), "
    "so the level correlation is mostly the shared multi-year trend: oil fell while recorded sales grew. "
    f"Store-day sales average {stats['payday_mean']:,.0f} on the 15th or the last day of the month, "
    f"{stats['day_after_payday_mean']:,.0f} on the next day, and {stats['other_day_mean']:,.0f} on other days. "
    "The calendar plot shows the clearer spike on the 1st and 2nd, after wages paid at month end, rather than on the 15th itself."
)
print(
    f"7. The earthquake on {stats['earthquake_date']} is a real shock in this sample. "
    f"Mean daily total sales were {stats['sales_14d_before_earthquake']:,.0f} in the 14 days before and "
    f"{stats['sales_14d_after_earthquake']:,.0f} in the 14 days after. "
    "A weekly seasonal model has no way to represent that break unless it is given an event flag."
)
print(
    f"Modelling subset: stores {stores} and families {families}. "
    "These are the largest series by unit sales. The full grid is 54 stores by 33 families, "
    "which is more than a CPU LSTM needs in order to compare methods. "
    f"Transactions correlate {stats['transactions_sales_corr']:.2f} with store-day sales, "
    "but they are recorded on the same day as sales, so they are not used as a predictor."
)

1. The training panel has 3,000,888 rows, 54 stores and 33 families, from 2013-01-01 to 2017-08-15. The only missing dates are 2013-12-25, 2014-12-25, 2015-12-25, 2016-12-25. Those are Christmas days, when the stores were closed, so the dates are absent rather than stored as zero.
2. Sales are highly concentrated. The five largest families account for 79% of unit sales. The busiest weekday in the company total is Sunday and the quietest is Thursday. Weekend days average 1.39 times a weekday. The strongest month is month 12 and the weakest is month 2.
3. Store type and cluster are not just labels. Mean total sales per store by type: C=10,962,317, E=14,955,609, B=18,157,580, D=19,504,628, A=39,227,092.
4. A matched holiday is not automatically a high-sales day. Mean store-day sales are 11,691 on ordinary days and 14,007 on national holidays (+19.8% versus an ordinary day). Christmas itself is missing from the file, so this comparison does not include the closure.
5. Promotions line up wi